### 샘플 오디오 추출

In [1]:
# pip install yt-dlp faster-whisper

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.3/180.3 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 63.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 60.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 MB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.8/38.8 MB 21.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.4/17.4 MB 134.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 9.8 MB/s eta 0:00:00


In [2]:
import os
from yt_dlp import YoutubeDL
from faster_whisper import WhisperModel

In [5]:
# 1. 설정
YOUTUBE_URL = "https://www.youtube.com/shorts/LXsTEqLbrRw"
OUTPUT_DIR = "voice_finetuning_test"
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [6]:
# 2. 유튜브에서 오디오 추출 (WAV 형식)
ydl_opts = {
    'format': 'bestaudio/best',
    'postprocessors': [{
        'key': 'FFmpegExtractAudio',
        'preferredcodec': 'wav',
        'preferredquality': '192',
    }],
    'outtmpl': f'{OUTPUT_DIR}/raw_audio.%(ext)s',
}

print("--- 오디오 다운로드 시작 ---")
with YoutubeDL(ydl_opts) as ydl:
    ydl.download([YOUTUBE_URL])
print("--- 오디오 다운로드 완료 ---")

--- 오디오 다운로드 시작 ---
[youtube] Extracting URL: https://www.youtube.com/shorts/LXsTEqLbrRw
[youtube] LXsTEqLbrRw: Downloading webpage


[youtube] LXsTEqLbrRw: Downloading android sdkless player API JSON
[youtube] LXsTEqLbrRw: Downloading web safari player API JSON


[youtube] LXsTEqLbrRw: Downloading m3u8 information


[info] LXsTEqLbrRw: Downloading 1 format(s): 251
[download] Destination: voice_finetuning_test/raw_audio.webm
[download] 100% of  605.68KiB in 00:00:00 at 822.57KiB/s 
[ExtractAudio] Destination: voice_finetuning_test/raw_audio.wav
Deleting original file voice_finetuning_test/raw_audio.webm (pass -k to keep)
--- 오디오 다운로드 완료 ---


In [7]:
# 3. Whisper를 이용한 텍스트 추출 및 라벨링
model_size = "large-v3"
model = WhisperModel(model_size, device="cuda", compute_type="float16")

print(f"--- Whisper {model_size} 모델 로드 및 텍스트 추출 시작 ---")
audio_path = os.path.join(OUTPUT_DIR, "raw_audio.wav")
segments, info = model.transcribe(audio_path, beam_size=5, language="ko")

# 결과 저장 (txt 파일 및 GPT-SoVITS 라벨 형식)
with open(os.path.join(OUTPUT_DIR, "transcription.txt"), "w", encoding="utf-8") as f:
    for segment in segments:
        line = f"[{segment.start:.2f}s -> {segment.end:.2f}s] {segment.text}"
        print(line)
        f.write(line + "\n")

print("--- 모든 작업이 완료되었습니다 ---")

model.bin:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/340 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

vocabulary.json: 0.00B [00:00, ?B/s]

--- Whisper large-v3 모델 로드 및 텍스트 추출 시작 ---
[0.00s -> 2.28s]  그걸 왜 저한테 물으세요?
[5.12s -> 5.26s]  네?
[8.56s -> 9.48s]  아저씨
[9.48s -> 14.44s]  지금 제가 경민이 죽인 것처럼 그러시잖아요.
[23.48s -> 25.64s]  죽는 건 무섭지 않아.
[25.64s -> 31.60s]  아, 언젠간 이 모든 것들이 다 끝난다는 게 다행이지 않아?
[34.18s -> 35.64s]  경민이가 그랬어요.
[45.04s -> 47.16s]  제가 죽여준 거예요.
[49.88s -> 53.84s]  경민이 걔요. 어차피 죽으래였다니까요.
[55.64s -> 56.42s]  자막 제공 및 자막 제공 및 광고를 포함하고 있습니다.
--- 모든 작업이 완료되었습니다 ---


In [ ]:
# !pip install pydub

In [6]:
from pydub import AudioSegment

# 설정
audio_path = "voice_finetuning_test/raw_audio.wav"
output_slice_dir = "voice_finetuning_test/sliced"
os.makedirs(output_slice_dir, exist_ok=True)

# 원본 오디오 로드
audio = AudioSegment.from_wav(audio_path)

# Whisper로 뽑은 타임스탬프와 대사 리스트
data = [
    (0.00, 2.28, "그걸 왜 저한테 물으세요?"),
    (5.12, 5.26, "네?"),
    (8.56, 9.48, "아저씨"),
    (9.48, 14.44, "지금 제가 경민이 죽인 것처럼 그러시잖아요."),
    (23.48, 25.64, "죽는 건 무섭지 않아."),
    (25.64, 31.60, "아, 언젠간 이 모든 것들이 다 끝난다는 게 다행이지 않아?"),
    (34.18, 35.64, "경민이가 그랬어요."),
    (45.04, 47.16, " 제가 죽여준 거예요."),
    (49.88, 53.84, "경민이 걔요. 어차피 죽을 애였다니까요.")
]

# 슬라이싱 및 라벨링 파일 생성
list_file_path = "voice_finetuning_test/list.txt"
with open(list_file_path, "w", encoding="utf-8") as f:
    for i, (start, end, text) in enumerate(data):
        # 밀리초(ms) 단위 사용
        start_ms = start * 1000
        end_ms = end * 1000
        
        # 오디오 자르기
        chunk = audio[start_ms:end_ms]
        filename = f"sample_{i:03d}.wav"
        chunk.export(os.path.join(output_slice_dir, filename), format="wav")
        
        # GPT-SoVITS 형식: 파일경로|화자이름|언어|대사
        # 언어: ko (한국어)
        line = f"sliced/{filename}|my_actor|ko|{text.strip()}\n"
        f.writelines(line)

print(f"슬라이싱 완료! {len(data)}개의 파일이 생성되었습니다.")

슬라이싱 완료! 9개의 파일이 생성되었습니다.


---

### 테스트

In [ ]:
# 1. 저장소 클론 및 이동
!git clone https://github.com/RVC-Boss/GPT-SoVITS.git
%cd GPT-SoVITS

In [ ]:
# 2. 필수 라이브러리 설치 (약 3~5분 소요)
!pip install -r requirements.txt
!pip install gradio

In [ ]:
# 3. 모델 파일 다운로드 (핵심 사전 학습 모델)
# (이 과정이 없으면 학습이 불가능합니다)
!python download_models.py

In [ ]:
# 4. WebUI 실행 (외부 접속용 share=True 옵션 포함)
# 실행 후 하단 로그에 "Running on public URL: https://xxxx.gradio.live"가 뜨면 클릭하세요!
!python webui.py --share